# Real-World Forensic Case Studies

This notebook applies the Proof of Funds risk pipeline to **four real criminal cases** with publicly known Bitcoin addresses. For each case, we:

1. Provide background and cite primary sources (DOJ, court filings)
2. Crawl the transaction neighborhood around known-bad addresses
3. Build the address graph with entity clustering
4. Compute all risk metrics (distance, direct exposure, haircut, aggregate)
5. Visualize taint propagation and compare to known law-enforcement findings

All data comes from **free public sources**: mempool.space API, GraphSense TagPacks, and DOJ press releases.

## 1. Setup

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))

from pof.cases import CASES, get_case
from pof.clustering import cluster_addresses, collapse_graph
from pof.explorer import Explorer, crawl_neighborhood
from pof.graph import build_graph, tainted_nodes
from pof.metrics import (
    AggregateWeights,
    aggregate_score,
    direct_exposure,
    distance_to_tainted,
    haircut_taint,
)
from pof.precompute import score_graph
from pof.tagpacks import discover_tagpack_files, load_tagpacks

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("pof.tagpacks").setLevel(logging.WARNING)

sns.set_theme(style="whitegrid", palette="muted")

DATA = Path("..") / "data"
TAGPACKS_DIR = DATA / "tagpacks" / "graphsense-tagpacks" / "packs"
CACHE_PATH = DATA / "cache" / "explorer.sqlite"
RESULTS_DIR = DATA / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load TagPacks

In [ ]:
tagpack_files = discover_tagpack_files(TAGPACKS_DIR)
tags = load_tagpacks(tagpack_files)
print(f"Loaded {len(tags):,} tagged BTC addresses from {len(tagpack_files)} files")
tags["category"].value_counts().head(10)

## 3. Helper: Investigate a Case

This function runs the full pipeline for a single case: crawl, cluster, build graph, score.

In [ ]:
explorer = Explorer(cache_path=CACHE_PATH)


def investigate_case(case_slug, hops=1, max_tx=25):
    """Run the full investigation pipeline for one case."""
    case = get_case(case_slug)
    print(f"\n{'='*60}")
    print(f"CASE: {case.name} ({case.date})")
    print(f"{'='*60}")
    print(f"Description: {case.description}")
    print(f"Seeds: {case.seed_addresses}")
    print(f"Sources: {case.sources}")

    # Crawl
    txs = crawl_neighborhood(
        explorer, case.seed_addresses, hops=hops, max_tx_per_addr=max_tx
    )
    print(f"\nCrawled {len(txs)} transactions")

    # Cluster addresses
    clusters = cluster_addresses(txs)
    n_entities = len(set(clusters.values()))
    n_addrs = len(clusters)
    print(f"Clustered {n_addrs} addresses into {n_entities} entities")

    # Build graphs (both address-level and entity-level)
    g_addr = build_graph(txs, tags=tags)
    g_entity = collapse_graph(g_addr, clusters)
    print(f"Address graph: {g_addr.number_of_nodes()} nodes, {g_addr.number_of_edges()} edges")
    print(f"Entity graph:  {g_entity.number_of_nodes()} nodes, {g_entity.number_of_edges()} edges")

    # Score both levels
    scores_addr = score_graph(g_addr)
    scores_entity = score_graph(g_entity)

    tainted = tainted_nodes(g_addr)
    print(f"\nTainted nodes in address graph: {len(tainted)}")

    return {
        "case": case,
        "txs": txs,
        "clusters": clusters,
        "g_addr": g_addr,
        "g_entity": g_entity,
        "scores_addr": scores_addr,
        "scores_entity": scores_entity,
    }

## 4. Case 1 — WannaCry Ransomware (2017)

WannaCry was a worldwide ransomware attack in May 2017 exploiting the EternalBlue vulnerability. The malware hardcoded three BTC payment addresses. Attributed to the Lazarus Group (North Korea) by the US, UK, and others.

**Source:** [DOJ Press Release](https://www.justice.gov/opa/pr/north-korean-regime-backed-programmer-charged-conspiracy-conduct-multiple-cyber-attacks-and)

In [ ]:
wc = investigate_case("wannacry")
print("\nTop 10 highest-risk addresses:")
wc["scores_addr"].nlargest(10, "score")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df = wc["scores_addr"]
axes[0].hist(df["score"], bins=30, edgecolor="black")
axes[0].set_xlabel("Aggregate Score")
axes[0].set_title("WannaCry: Score Distribution")

axes[1].hist(df["distance"].replace(-1, np.nan).dropna(), bins=15, edgecolor="black")
axes[1].set_xlabel("Distance (hops)")
axes[1].set_title("WannaCry: Distance to Tainted")

axes[2].hist(df["haircut"], bins=30, edgecolor="black")
axes[2].set_xlabel("Haircut Taint")
axes[2].set_title("WannaCry: Haircut Distribution")

plt.tight_layout()
plt.show()

## 5. Case 2 — Twitter Hack (2020)

On July 15, 2020, attackers compromised high-profile Twitter accounts (Obama, Musk, Apple) and posted a BTC doubling scam, collecting ~$120k in BTC.

**Source:** [DOJ Complaint](https://www.justice.gov/opa/press-release/file/1300261/download)

In [ ]:
tw = investigate_case("twitter_hack")
print("\nTop 10 highest-risk addresses:")
tw["scores_addr"].nlargest(10, "score")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df = tw["scores_addr"]
axes[0].hist(df["score"], bins=30, edgecolor="black")
axes[0].set_xlabel("Aggregate Score")
axes[0].set_title("Twitter Hack: Score Distribution")

axes[1].hist(df["distance"].replace(-1, np.nan).dropna(), bins=15, edgecolor="black")
axes[1].set_xlabel("Distance (hops)")
axes[1].set_title("Twitter Hack: Distance to Tainted")

axes[2].hist(df["haircut"], bins=30, edgecolor="black")
axes[2].set_xlabel("Haircut Taint")
axes[2].set_title("Twitter Hack: Haircut Distribution")

plt.tight_layout()
plt.show()

## 6. Case 3 — Colonial Pipeline / DarkSide (2021)

In May 2021, the DarkSide ransomware group attacked Colonial Pipeline, causing fuel shortages across the US East Coast. Colonial paid ~75 BTC (~$4.4M). The DOJ later recovered ~63.7 BTC.

**Source:** [DOJ Seizure Announcement](https://www.justice.gov/opa/pr/department-justice-seizes-23-million-cryptocurrency-paid-ransomware-extortionists-darkside)

In [ ]:
cp = investigate_case("colonial_pipeline")
print("\nTop 10 highest-risk addresses:")
cp["scores_addr"].nlargest(10, "score")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df = cp["scores_addr"]
axes[0].hist(df["score"], bins=30, edgecolor="black")
axes[0].set_xlabel("Aggregate Score")
axes[0].set_title("Colonial Pipeline: Score Distribution")

axes[1].hist(df["distance"].replace(-1, np.nan).dropna(), bins=15, edgecolor="black")
axes[1].set_xlabel("Distance (hops)")
axes[1].set_title("Colonial Pipeline: Distance to Tainted")

axes[2].hist(df["haircut"], bins=30, edgecolor="black")
axes[2].set_xlabel("Haircut Taint")
axes[2].set_title("Colonial Pipeline: Haircut Distribution")

plt.tight_layout()
plt.show()

## 7. Case 4 — Bitfinex Hack (2016)

In August 2016, ~119,756 BTC were stolen from the Bitfinex exchange. In February 2022, Ilya Lichtenstein and Heather Morgan were arrested and ~94,000 BTC were seized — the largest financial seizure in DOJ history.

**Source:** [DOJ Arrest Announcement](https://www.justice.gov/opa/pr/two-arrested-alleged-conspiracy-launder-45-billion-stolen-cryptocurrency)

In [ ]:
bf = investigate_case("bitfinex_hack")
print("\nTop 10 highest-risk addresses:")
bf["scores_addr"].nlargest(10, "score")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df = bf["scores_addr"]
axes[0].hist(df["score"], bins=30, edgecolor="black")
axes[0].set_xlabel("Aggregate Score")
axes[0].set_title("Bitfinex Hack: Score Distribution")

axes[1].hist(df["distance"].replace(-1, np.nan).dropna(), bins=15, edgecolor="black")
axes[1].set_xlabel("Distance (hops)")
axes[1].set_title("Bitfinex: Distance to Tainted")

axes[2].hist(df["haircut"], bins=30, edgecolor="black")
axes[2].set_xlabel("Haircut Taint")
axes[2].set_title("Bitfinex: Haircut Distribution")

plt.tight_layout()
plt.show()

## 8. Cross-Case Comparison

Compare the four cases side by side: graph size, clustering effect, and score distributions.

In [ ]:
results = {"wannacry": wc, "twitter_hack": tw, "colonial_pipeline": cp, "bitfinex_hack": bf}

comparison = []
for slug, r in results.items():
    sa = r["scores_addr"]
    se = r["scores_entity"]
    comparison.append({
        "Case": r["case"].name,
        "Transactions": len(r["txs"]),
        "Addresses": r["g_addr"].number_of_nodes(),
        "Entities": r["g_entity"].number_of_nodes(),
        "Cluster Reduction": f"{(1 - r['g_entity'].number_of_nodes() / max(r['g_addr'].number_of_nodes(), 1)) * 100:.1f}%",
        "Tainted Nodes": len(tainted_nodes(r["g_addr"])),
        "Mean Score (addr)": f"{sa['score'].mean():.1f}",
        "Median Score (addr)": f"{sa['score'].median():.1f}",
        "Max Score (addr)": f"{sa['score'].max():.1f}",
        "Mean Score (entity)": f"{se['score'].mean():.1f}",
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (slug, r) in zip(axes.flat, results.items()):
    df_a = r["scores_addr"]
    df_e = r["scores_entity"]
    ax.hist(df_a["score"], bins=25, alpha=0.6, label="Address-level", edgecolor="black")
    ax.hist(df_e["score"], bins=25, alpha=0.6, label="Entity-level", edgecolor="black")
    ax.set_title(r["case"].name)
    ax.set_xlabel("Score")
    ax.legend()

plt.suptitle("Address-Level vs Entity-Level Score Distributions", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Taint Path Tracing

For each case, find the highest-scoring **untagged** address and trace the shortest path from a tainted source to it. This demonstrates how the risk propagated through the graph.

In [ ]:
for slug, r in results.items():
    g = r["g_addr"]
    df = r["scores_addr"]
    untagged = df[df["severity"] == 0.0]
    if untagged.empty:
        print(f"\n{r['case'].name}: All addresses are tagged, skipping path trace.")
        continue

    target = untagged["score"].idxmax()
    tainted = [n for n, d in g.nodes(data=True) if d.get("severity", 0) > 0]

    print(f"\n{'='*60}")
    print(f"{r['case'].name}: Tracing path to highest-risk untagged address")
    print(f"Target: {target} (score={df.loc[target, 'score']:.1f})")

    best_path = None
    for src in tainted:
        try:
            path = nx.shortest_path(g, src, target)
            if best_path is None or len(path) < len(best_path):
                best_path = path
        except nx.NetworkXNoPath:
            continue

    if best_path:
        print(f"Shortest path ({len(best_path)-1} hops):")
        for i, addr in enumerate(best_path):
            sev = g.nodes[addr].get('severity', 0)
            label = g.nodes[addr].get('label', '')
            tag_info = f" [severity={sev}, label={label}]" if sev > 0 else ""
            if i < len(best_path) - 1:
                edge = g[best_path[i]][best_path[i+1]]
                print(f"  {addr}{tag_info}  --({edge['value_sat']:,} sat)-->")
            else:
                print(f"  {addr}{tag_info}")
    else:
        print("  No directed path found from any tainted node.")

## 10. Entity Clustering Analysis

Examine the effect of clustering: how many addresses collapse into each entity?

In [ ]:
from collections import Counter

for slug, r in results.items():
    clusters = r["clusters"]
    entity_sizes = Counter(clusters.values())
    sizes = sorted(entity_sizes.values(), reverse=True)
    print(f"\n{r['case'].name}:")
    print(f"  Total addresses: {len(clusters)}")
    print(f"  Total entities:  {len(entity_sizes)}")
    print(f"  Largest entity:  {sizes[0]} addresses")
    print(f"  Entities with >1 addr: {sum(1 for s in sizes if s > 1)}")
    print(f"  Top 5 entity sizes: {sizes[:5]}")

## 11. Save Results

Export all case scores as Parquet files for use by the dashboard and validation notebook.

In [ ]:
for slug, r in results.items():
    path = RESULTS_DIR / f"case_{slug}_scores.parquet"
    r["scores_addr"].to_parquet(path)
    print(f"Saved {slug}: {len(r['scores_addr'])} addresses -> {path}")

    path_e = RESULTS_DIR / f"case_{slug}_entity_scores.parquet"
    r["scores_entity"].to_parquet(path_e)
    print(f"Saved {slug} (entity): {len(r['scores_entity'])} entities -> {path_e}")

print("\nDone. All case results exported.")

## 12. Limitations

- **Bounded crawl:** we fetch at most ~25 tx per address and 1 hop, so deep laundering chains are not fully explored.
- **Entity clustering** uses only the co-spend and basic change-address heuristics. Sophisticated users can defeat these.
- **TagPack coverage** depends on the GraphSense community dataset. Addresses not tagged will have severity 0 even if illicit.
- **Severity weights are illustrative** and not calibrated to any specific compliance regime.
- **Bitcoin only:** the pipeline does not support Ethereum or other chains.